# 💬 PHILIA — Text Emotion Fine-Tuning (Optuna)
**Model:** `roberta-base` → fine-tuned on MELD conversation text  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Optuna:** 10-trial hyperparameter search → final training with best params  
**Output:** saved to Google Drive → download and drop into PHILIA

**Runtime:** Set to `GPU` → Runtime > Change runtime type > T4 GPU

---
## Workflow (2 phases — both are needed!)
| Phase | Cells | What happens |
|-------|-------|--------------|
| **Phase 1 — Optuna search** | 1–9 | Runs 10 short trials to find the best learning rate, batch size, etc. |
| **Phase 2 — Final training** | 10–13 | Trains a *fresh* model end-to-end using those best hyperparameters. |

> ⚠️ **Both phases are required.** Phase 1 finds *what* to train with. Phase 2 does the real training.

## Key Improvements Over Baseline
- **Focal Loss** replaces plain weighted cross-entropy → focuses on hard/minority emotion classes
- **10 Optuna trials** (up from 8) → text is fast; more search = better hyperparams
- **Context window (256 tokens)** instead of 128 → MELD utterances benefit from dialogue context
- **Label smoothing (0.1)** → prevents overconfidence on noisy MELD labels
- **Macro F1 tracking** → monitor per-class performance to catch minority-class failures
- **Gradient checkpointing** → lower memory footprint, stable training

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate scikit-learn optuna

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/text_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Download MELD text CSVs from GitHub ────────────────────────────
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv -O /content/train.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv   -O /content/dev.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv  -O /content/test.csv
print('Downloaded.')

In [ ]:
# ── Cell 5: Load and prepare data ─────────────────────────────────────────
# Enhancement: Include speaker name + dialogue context to improve accuracy.
# Within each dialogue, we prepend the previous utterance as context so the
# model can use conversational cues (e.g. sarcasm detected from reply pattern).
import pandas as pd
from datasets import Dataset

EMOTION_MAP = {
    'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',
    'joy':      'happy',
    'neutral':  'neutral',
    'sadness':  'sad',
    'surprise': 'surprise',
}
LABELS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

def load_split(path, name, add_context=True):
    df = pd.read_csv(path)
    df['Emotion']   = df['Emotion'].str.lower()
    df['canonical'] = df['Emotion'].map(EMOTION_MAP)
    df = df[df['canonical'].notna()]
    df['label'] = df['canonical'].map(LABEL2ID)
    df = df[['Utterance', 'Speaker', 'Dialogue_ID', 'Utterance_ID', 'label', 'canonical']].rename(
        columns={'Utterance': 'text'}
    )
    df = df[df['text'].notna() & (df['text'].str.strip() != '')]

    if add_context:
        # Prepend previous utterance as dialogue context
        df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])
        prev_text = df.groupby('Dialogue_ID')['text'].shift(1).fillna('')
        df['text'] = df.apply(
            lambda r: f"{r['Speaker']}: {r['text']}" if prev_text[r.name] == ''
            else f"Context: {prev_text[r.name]} | {r['Speaker']}: {r['text']}",
            axis=1
        )

    print(f'{name}: {len(df)} samples | dist: {dict(df["canonical"].value_counts())}')
    return df.reset_index(drop=True)

train_df = load_split('/content/train.csv', 'TRAIN')
val_df   = load_split('/content/dev.csv',   'VAL')
test_df  = load_split('/content/test.csv',  'TEST')

train_ds = Dataset.from_pandas(train_df[['text', 'label']])
val_ds   = Dataset.from_pandas(val_df[['text', 'label']])
test_ds  = Dataset.from_pandas(test_df[['text', 'label']])
print('Datasets created.')

In [ ]:
# ── Cell 6: Tokenize ──────────────────────────────────────────────────────
# Extended to 256 tokens (from 128) to fit context-enriched utterances
from transformers import AutoTokenizer

MODEL_CHECKPOINT = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

MAX_LENGTH = 256   # fits speaker + context + utterance comfortably

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])
test_ds  = test_ds.map(tokenize,  batched=True, remove_columns=['text'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Tokenized.')

In [ ]:
# ── Cell 7: Metrics · Class Weights ───────────────────────────────────────
import evaluate, torch, torch.nn as nn, numpy as np
from sklearn.metrics import f1_score
from collections import Counter
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc    = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1     = f1_score(labels, preds, average='weighted')
    f1_mac = f1_score(labels, preds, average='macro')   # reveals per-class struggles
    return {'accuracy': acc, 'f1': f1, 'f1_macro': f1_mac}

# Class weights — neutral ≈ 47% of MELD train; boost minority classes
label_counts  = Counter(train_df['label'].tolist())
total         = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (len(LABELS) * label_counts.get(i, 1)) for i in range(len(LABELS))],
    dtype=torch.float,
)
print('Class weights:', {ID2LABEL[i]: f'{w:.3f}' for i, w in enumerate(class_weights)})

In [ ]:
# ── Cell 8: model_init · FocalLossTrainer · hp_space ──────────────────────
# Focal Loss vs plain weighted CE:
# → down-weights easy majority-class examples (neutral) and focuses
#   gradient on hard minority classes (fear, disgust)
import optuna
from transformers import AutoModelForSequenceClassification

optuna.logging.set_verbosity(optuna.logging.WARNING)

FOCAL_GAMMA = 2.0   # γ=2 is a good default

def focal_loss(logits, labels, gamma=FOCAL_GAMMA):
    """Focal loss with class weighting."""
    weights = class_weights.to(logits.device)
    ce_loss = nn.functional.cross_entropy(logits, labels, weight=weights, reduction='none')
    pt = torch.exp(-ce_loss)         # probability of the correct class
    return ((1 - pt) ** gamma * ce_loss).mean()

def model_init(trial=None):
    """Fresh RoBERTa for every Optuna trial — required by hyperparameter_search."""
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(LABELS),
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True,
    )

class FocalLossTrainer(Trainer):
    """Trainer with Focal Loss to handle class imbalance better than plain CE."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def hp_space(trial):
    """Search space for Optuna — text training is fast so we can afford more trials."""
    return {
        'learning_rate':               trial.suggest_float('learning_rate', 5e-6, 8e-4, log=True),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [16, 32]),
        'num_train_epochs':            trial.suggest_int('num_train_epochs', 3, 10),
        'weight_decay':                trial.suggest_float('weight_decay', 1e-4, 0.15, log=True),
        'warmup_ratio':                trial.suggest_float('warmup_ratio', 0.03, 0.25),
    }

print('model_init, FocalLossTrainer and hp_space defined.')

---
## Phase 1 — Optuna Hyperparameter Search
Runs **10 short trials** (text is fast!) to discover the best learning rate, batch size, epochs, weight decay, and warmup ratio.  
Optuna prunes poor trials early to save time.

> **This does NOT produce your final model.** It only finds the best settings for Phase 2.

In [ ]:
# ── Cell 9: Optuna Hyperparameter Search (10 trials) ───────────────────────
# PHASE 1 — finds best hyperparameters, does NOT save a final model
# Text training is faster than audio, so we use 10 trials for better coverage.

search_args = TrainingArguments(
    output_dir='/content/optuna_search',
    eval_strategy='epoch',
    save_strategy='no',          # no checkpoints during search
    logging_steps=100,
    fp16=True,
    gradient_checkpointing=True, # saves GPU memory
    report_to='none',
)

search_trainer = FocalLossTrainer(
    model_init=model_init,       # model_init is REQUIRED here (not model=)
    args=search_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print('Running Optuna search — 10 trials × up to 10 epochs each...')
print('(Optuna will prune bad trials early — total time ~20-40 min on T4)')
best_run = search_trainer.hyperparameter_search(
    direction='maximize',
    backend='optuna',
    hp_space=hp_space,
    n_trials=10,
    compute_objective=lambda m: m['eval_f1'],   # optimise for weighted F1
)

print(f'\n✅ Phase 1 complete!')
print(f'   Best trial  : {best_run.run_id}')
print(f'   eval_f1     : {best_run.objective:.4f}')
print('   Best hyperparameters:')
for k, v in best_run.hyperparameters.items():
    print(f'     {k}: {v}')

---
## Phase 2 — Final Training with Best Hyperparameters
Now that we know the best settings, we train a **fresh model all the way through** using those settings.
This is where the real production model is produced and saved.  
> **This is NOT a duplicate.** Phase 1 = tuning, Phase 2 = actual full training.

In [ ]:
# ── Cell 10: Final Training (PHASE 2) ─────────────────────────────────────
# Trains a BRAND NEW model using the best hyperparameters found by Optuna.
# This is the model that will be saved and used in PHILIA.

hp = best_run.hyperparameters

final_args = TrainingArguments(
    output_dir='/content/roberta_text_emotion',
    num_train_epochs=hp.get('num_train_epochs', 6),
    per_device_train_batch_size=hp.get('per_device_train_batch_size', 32),
    per_device_eval_batch_size=64,
    warmup_ratio=hp.get('warmup_ratio', 0.1),
    learning_rate=hp.get('learning_rate', 2e-5),
    weight_decay=hp.get('weight_decay', 0.01),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    report_to='none',
    label_smoothing_factor=0.1,  # prevents overconfidence on noisy MELD labels
)

final_trainer = FocalLossTrainer(
    model=model_init(),              # fresh model (no trial= arg needed here)
    args=final_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Starting Phase 2 — final training with best hyperparameters...')
final_trainer.train()

In [ ]:
# ── Cell 11: Evaluate on test set ─────────────────────────────────────────
from sklearn.metrics import classification_report

test_results = final_trainer.evaluate(test_ds)
print('Test results:', test_results)

pred_output = final_trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
print(classification_report(
    pred_output.label_ids, preds,
    target_names=LABELS,
    digits=3
))

In [ ]:
# ── Cell 12: Save to Google Drive ─────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
final_trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 13: (Optional) Push to Hugging Face Hub ──────────────────────────
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# final_trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-text-emotion')
# tokenizer.push_to_hub('YOUR_HF_USERNAME/philia-text-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')